# Sensi v2 — Curriculum-Based Test-Time Learning Demo

**Sensi: Learn One Thing at a Time**

This notebook runs the SensiLLM agent on an [ARC-AGI-3](https://three.arcprize.org/) game.
The agent uses a two-player cooperative strategy (Observer + Actor) with persistent
learning stored in SQLite, driven by a curriculum state machine and dynamic LLM-as-judge scoring.

**What you need:**
- An ARC-AGI-3 API key → [three.arcprize.org](https://three.arcprize.org/)
- A Gemini API key → [aistudio.google.com](https://aistudio.google.com/)

📄 [Paper](https://arxiv.org/abs/2603.17683) · 💻 [GitHub](https://github.com/arjmandi/sensi)

## 1. Clone the repo & install dependencies

In [ ]:
!git clone https://github.com/arjmandi/sensi.git
%cd sensi

In [ ]:
!pip install -q \
    'dotenv>=0.9.9' \
    'dspy==2.5.32' \
    'dspy-ai==2.5.32' \
    'google-auth>=2.0.0' \
    'google-cloud-aiplatform>=1.38.0' \
    'google-genai>=0.3.0' \
    'litellm==1.51.0' \
    'pillow>=11.2.1' \
    'pydantic>=2.11.7' \
    'requests>=2.32.4'

## 2. Set API keys

Enter your keys below. They are stored only in this runtime session.

In [ ]:
import os
from getpass import getpass

os.environ["ARC_API_KEY"] = getpass("ARC-AGI-3 API key: ")
os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

# Server config (defaults for the public ARC-AGI-3 API)
os.environ["SCHEME"] = "https"
os.environ["HOST"] = "three.arcprize.org"
os.environ["PORT"] = "443"
os.environ["RECORDINGS_DIR"] = "recordings"
os.environ["DEBUG"] = "False"

print("Keys set.")

## 3. Run SensiLLM

Run the agent against a single game (`ls20` by default). Change the `--game` flag
to try other ARC-AGI-3 levels, or remove it to run against all available games.

In [ ]:
!python main.py --agent=sensillm --game=ls20

## 4. Inspect the learning state

Sensi persists its entire cognitive state in SQLite. Let's peek at what it learned.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("agent_state.db")

print("=== Learning Items ===")
df_items = pd.read_sql_query(
    "SELECT item_name, state, current_sense_score, threshold FROM items_to_learn",
    conn,
)
display(df_items)

print("\n=== Facts (confirmed knowledge) ===")
df_facts = pd.read_sql_query(
    "SELECT item_name FROM items_to_learn WHERE state = 'fact'",
    conn,
)
display(df_facts)

print("\n=== Last 10 actions ===")
df_game = pd.read_sql_query(
    "SELECT turn_id, prev_action, prev_decision_type, game_state FROM game ORDER BY turn_id DESC LIMIT 10",
    conn,
)
display(df_game)

conn.close()

## 5. Run against all games (optional)

Remove the `--game` filter to run SensiLLM against every available ARC-AGI-3 game.
Each game runs in its own thread.

In [ ]:
# Uncomment to run against all games:
# !python main.py --agent=sensillm

---

**Paper:** [Sensi: Learn One Thing at a Time — Curriculum-Based Test-Time Learning for LLM Game Agents](https://arxiv.org/abs/2603.17683)

**GitHub:** [arjmandi/sensi](https://github.com/arjmandi/sensi)

If you found this useful, give the repo a ⭐!